<a href="https://colab.research.google.com/github/shahidabatool/App2025/blob/master/llm_matching_markets_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================
# phase 1 setup and data preparation
# =========================================

import pandas as pd
import os

# clone repo only if not already present
if not os.path.exists("llm-matching-markets-project"):
    !git clone https://github.com/NimrahAltafAdam/llm-matching-markets-project.git

# set base path
base_path = "llm-matching-markets-project"

# load dataset from repo
df = pd.read_csv(os.path.join(base_path, "data", "10_ic_processed.csv"))

print("shape:", df.shape)

print("\ncolumns:")
print(df.columns.tolist())

print("\nfirst row:")
print(df.iloc[0])

Cloning into 'llm-matching-markets-project'...
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 33 (delta 11), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (33/33), 63.13 KiB | 3.95 MiB/s, done.
Resolving deltas: 100% (11/11), done.
shape: (100, 19)

columns:
['pref_type', 'n_man', 'n_woman', 'combined_pref_json', 'man_pref_string', 'woman_pref_string', 'men_opt', 'women_opt', 'lattice', 'random_1', 'random_3', 'random_10', 'random', 'level1_q', 'level1_a', 'level2_q', 'level2_a', 'level2n_q', 'level2n_a']

first row:
pref_type                                             impartial culture
n_man                                                                10
n_woman                                                              10
combined_pref_json    {\nM: {\nM1: [W10,W1,W3,W6,W2,W4,W9,W8,W7,W5],...
man_pref_string       10,1,3,6,2,4,9,8,7,5\n8,3,10,6,2,5,4,7,1,9\n4

In [ ]:
# =========================================
# step 1 load preferences and ground truth
# =========================================

row = df.iloc[0]

man_prefs = row["man_pref_string"]
woman_prefs = row["woman_pref_string"]
ground_truth = row["men_opt"]

print("=== MEN PREFERENCES ===")
print(man_prefs)

print("\n=== WOMEN PREFERENCES ===")
print(woman_prefs)

print("\n=== GROUND TRUTH MATCHING ===")
print(ground_truth)

=== MEN PREFERENCES ===
10,1,3,6,2,4,9,8,7,5
8,3,10,6,2,5,4,7,1,9
4,9,3,8,5,2,6,7,1,10
2,4,6,8,7,1,5,10,3,9
3,8,5,2,7,6,10,4,9,1
7,1,9,10,3,6,5,4,2,8
6,4,1,8,9,7,3,10,5,2
8,5,3,10,9,1,7,6,2,4
8,4,1,9,5,7,3,2,6,10
2,5,1,3,7,6,10,4,9,8

=== WOMEN PREFERENCES ===
2,8,9,10,5,7,1,4,6,3
2,7,3,1,8,9,6,10,5,4
4,10,2,6,5,8,1,9,3,7
6,2,9,1,5,8,3,4,10,7
8,4,5,3,6,7,1,10,9,2
6,9,5,1,4,2,8,3,7,10
10,9,2,1,4,8,3,7,5,6
10,2,9,4,6,7,1,8,5,3
8,6,4,3,7,10,5,9,2,1
6,4,7,5,8,9,10,2,3,1

=== GROUND TRUTH MATCHING ===
[[M1, W10],[M2, W8],[M3, W9],[M4, W6],[M5, W3],[M6, W7],[M7, W1],[M8, W5],[M9, W4],[M10, W2],]


In [ ]:
# =========================================
# step 2 parse raw preference data
# =========================================

# convert raw strings into structured lists

men_prefs_raw = row["man_pref_string"].split("\n")
women_prefs_raw = row["woman_pref_string"].split("\n")

men_prefs = [list(map(int, line.split(","))) for line in men_prefs_raw]
women_prefs = [list(map(int, line.split(","))) for line in women_prefs_raw]

In [ ]:
# =========================================
# step 3 build formatted prompt
# =========================================

# build readable preference strings in paper-style format

men_text = ""
for i, prefs in enumerate(men_prefs):
    men_text += f"M{i+1}: [" + ",".join([f"W{w}" for w in prefs]) + "]\n"

women_text = ""
for i, prefs in enumerate(women_prefs):
    women_text += f"W{i+1}: [" + ",".join([f"M{m}" for m in prefs]) + "]\n"

prompt = f"""
You are an intelligent assistant who is an expert in algorithms. Consider the following instance of the two-sided matching problem, where 10 men are to be matched with 10 women.

Here are the preference lists for all individuals:

<preferences>
{{
M: {{
{men_text}
}},
W: {{
{women_text}
}}
}}
</preferences>

Your task is to find the proposer-optimal stable matching.

Return ONLY the final matching.
Do NOT provide Python code.
Do NOT explain your reasoning.
Do NOT describe the algorithm.
Your final response must contain only one JSON object enclosed in <answer></answer> tags.

<answer>
{{
  "M1": "<woman matched with M1>",
  "M2": "<woman matched with M2>",
  "M3": "<woman matched with M3>",
  "M4": "<woman matched with M4>",
  "M5": "<woman matched with M5>",
  "M6": "<woman matched with M6>",
  "M7": "<woman matched with M7>",
  "M8": "<woman matched with M8>",
  "M9": "<woman matched with M9>",
  "M10": "<woman matched with M10>"
}}
</answer>

Make sure that each man is matched with exactly ONE woman and each woman is matched with exactly ONE man.
"""

print(prompt)


You are an intelligent assistant who is an expert in algorithms. Consider the following instance of the two-sided matching problem, where 10 men are to be matched with 10 women.

Here are the preference lists for all individuals:

<preferences>
{
M: {
M1: [W10,W1,W3,W6,W2,W4,W9,W8,W7,W5]
M2: [W8,W3,W10,W6,W2,W5,W4,W7,W1,W9]
M3: [W4,W9,W3,W8,W5,W2,W6,W7,W1,W10]
M4: [W2,W4,W6,W8,W7,W1,W5,W10,W3,W9]
M5: [W3,W8,W5,W2,W7,W6,W10,W4,W9,W1]
M6: [W7,W1,W9,W10,W3,W6,W5,W4,W2,W8]
M7: [W6,W4,W1,W8,W9,W7,W3,W10,W5,W2]
M8: [W8,W5,W3,W10,W9,W1,W7,W6,W2,W4]
M9: [W8,W4,W1,W9,W5,W7,W3,W2,W6,W10]
M10: [W2,W5,W1,W3,W7,W6,W10,W4,W9,W8]

},
W: {
W1: [M2,M8,M9,M10,M5,M7,M1,M4,M6,M3]
W2: [M2,M7,M3,M1,M8,M9,M6,M10,M5,M4]
W3: [M4,M10,M2,M6,M5,M8,M1,M9,M3,M7]
W4: [M6,M2,M9,M1,M5,M8,M3,M4,M10,M7]
W5: [M8,M4,M5,M3,M6,M7,M1,M10,M9,M2]
W6: [M6,M9,M5,M1,M4,M2,M8,M3,M7,M10]
W7: [M10,M9,M2,M1,M4,M8,M3,M7,M5,M6]
W8: [M10,M2,M9,M4,M6,M7,M1,M8,M5,M3]
W9: [M8,M6,M4,M3,M7,M10,M5,M9,M2,M1]
W10: [M6,M4,M7,M5,M8,M9,M10,M2,M3,

In [ ]:
# =========================================
# step 4 install and load api key securely
# =========================================

!pip install -q langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.7 MB/s eta 0:00:00


In [ ]:
from getpass import getpass

GROQ_API_KEY = getpass()

··········


In [ ]:
import os
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [ ]:
# =========================================
# phase 2 helper functions
# =========================================

def prefers(preference_list, a, b):
    return preference_list.index(a) < preference_list.index(b)

# =========================================
# step 1 stability check function (fixed)
# =========================================

def convert_prefs_to_dict(prefs, side_prefix, other_prefix):
    # if already a dict, return as is
    if isinstance(prefs, dict):
        return prefs

    converted = {}
    for i, pref_list in enumerate(prefs, start=1):
        agent = f"{side_prefix}{i}"
        converted[agent] = [f"{other_prefix}{x}" for x in pref_list]
    return converted


def check_stability(matching, men_prefs, women_prefs):
    # convert list-based prefs into dict format if needed
    men_prefs = convert_prefs_to_dict(men_prefs, "M", "W")
    women_prefs = convert_prefs_to_dict(women_prefs, "W", "M")

    reverse_matching = {w: m for m, w in matching.items()}
    blocking_pairs = []

    for m in men_prefs:
        current_w = matching.get(m)
        if current_w is None:
            continue

        m_pref_list = men_prefs[m]

        if current_w not in m_pref_list:
            continue

        current_w_index = m_pref_list.index(current_w)
        preferred_women = m_pref_list[:current_w_index]

        for w in preferred_women:
            current_m_for_w = reverse_matching.get(w)
            if current_m_for_w is None:
                continue

            w_pref_list = women_prefs[w]

            if m in w_pref_list and current_m_for_w in w_pref_list:
                if w_pref_list.index(m) < w_pref_list.index(current_m_for_w):
                    blocking_pairs.append((m, w))

    return len(blocking_pairs) == 0, blocking_pairs

In [ ]:
# =========================================
# step 2 validity check function
# =========================================

def check_validity(llm_matching, expected_size=10):
    # must have exactly expected_size men
    if len(llm_matching) != expected_size:
        return False, "wrong number of men in output"

    # all assigned partners
    assigned_women = list(llm_matching.values())

    # each woman must be unique
    if len(set(assigned_women)) != expected_size:
        return False, "duplicate women assigned"

    # check label format roughly
    expected_men = {f"M{i}" for i in range(1, expected_size + 1)}
    expected_women = {f"W{i}" for i in range(1, expected_size + 1)}

    if set(llm_matching.keys()) != expected_men:
        return False, "men labels are incorrect or incomplete"

    if not set(assigned_women).issubset(expected_women):
        return False, "women labels are incorrect"

    return True, "valid one-to-one matching"

In [ ]:
# =========================================
# step 3 response parsing and ground truth functions
# =========================================

import re

def parse_ground_truth_pairs(ground_truth_string):
    pairs = re.findall(r'\[M(\d+),\s*W(\d+)\]', ground_truth_string)
    return {f"M{m}": f"W{w}" for m, w in pairs}


def exact_match_with_ground_truth(llm_matching, ground_truth_string):
    ground_truth_dict = parse_ground_truth_pairs(ground_truth_string)
    return llm_matching == ground_truth_dict, ground_truth_dict

In [ ]:
# =========================================
# step 4 evaluation pipeline
# =========================================

def evaluate_matching(llm_matching, men_prefs, women_prefs, ground_truth, expected_size=10):
    result = {
        "parsed_successfully": llm_matching is not None,
        "is_valid": False,
        "is_stable": False,
        "blocking_pairs": [],
        "exact_match": False,
        "ground_truth_dict": None
    }

    if llm_matching is None:
        return result

    is_valid, validity_message = check_validity(llm_matching, expected_size=expected_size)
    result["is_valid"] = is_valid
    result["validity_message"] = validity_message

    if not is_valid:
        return result

    is_stable, blocking_pairs = check_stability(llm_matching, men_prefs, women_prefs)
    result["is_stable"] = is_stable
    result["blocking_pairs"] = blocking_pairs

    exact_match, ground_truth_dict = exact_match_with_ground_truth(llm_matching, ground_truth)
    result["exact_match"] = exact_match
    result["ground_truth_dict"] = ground_truth_dict

    return result

In [ ]:
def print_evaluation_summary(result):
    print("parsed:", result["parsed_successfully"])
    print("valid:", result["is_valid"])
    print("stable:", result["is_stable"])
    print("exact match:", result["exact_match"])

    if "validity_message" in result:
        print("validity message:", result["validity_message"])

    print("blocking pairs count:", len(result["blocking_pairs"]))
    if result["blocking_pairs"]:
        print("blocking pairs:", result["blocking_pairs"])

In [ ]:
import re
import json

def normalize_agent_label(label, prefix):
    label = str(label).strip().upper()
    match = re.match(rf"{prefix}\s*(\d+)$", label)
    if match:
        return f"{prefix}{match.group(1)}"
    return label


def normalize_matching_dict(matching):
    normalized = {}
    for k, v in matching.items():
        man = normalize_agent_label(k, "M")
        woman = normalize_agent_label(v, "W")
        normalized[man] = woman
    return normalized


# def extract_matching_from_response(raw_output):
#     # first try to extract json inside <answer> tags
#     answer_match = re.search(r"<answer>\s*(\{[\s\S]*?\})\s*</answer>", raw_output, re.IGNORECASE)

#     candidate_json = None

#     if answer_match:
#         candidate_json = answer_match.group(1)
#     else:
#         # fallback to any json-like object in the response
#         matches = re.findall(r"\{[\s\S]*?\}", raw_output)
#         if matches:
#             candidate_json = matches[-1]

#     if not candidate_json:
#         return None, False, "No JSON found"

#     try:
#         parsed = json.loads(candidate_json)
#         normalized = normalize_matching_dict(parsed)
#         return normalized, True, "Parsed successfully"
#     except Exception as e:
#         return None, False, f"JSON parse error: {str(e)}"

def extract_matching_from_response(raw_output):
    import re
    import json

    # try 1: <answer> ... </answer>
    answer_match = re.search(
        r"<answer>\s*(\{[\s\S]*?\})\s*</answer>",
        raw_output,
        re.IGNORECASE
    )
    if answer_match:
        candidate_json = answer_match.group(1)
        try:
            parsed = json.loads(candidate_json)
            normalized = {str(k).strip().upper(): str(v).strip().upper() for k, v in parsed.items()}
            return normalized, True, "Parsed from <answer> tags"
        except Exception as e:
            return None, False, f"JSON parse error in <answer>: {str(e)}"

    # try 2: ```json ... ```
    codeblock_match = re.search(
        r"```json\s*(\{[\s\S]*?\})\s*```",
        raw_output,
        re.IGNORECASE
    )
    if codeblock_match:
        candidate_json = codeblock_match.group(1)
        try:
            parsed = json.loads(candidate_json)
            normalized = {str(k).strip().upper(): str(v).strip().upper() for k, v in parsed.items()}
            return normalized, True, "Parsed from code block"
        except Exception as e:
            return None, False, f"JSON parse error in code block: {str(e)}"

    # try 3: fallback JSON object
    matches = re.findall(r"\{[\s\S]*?\}", raw_output)
    for candidate_json in reversed(matches):
        try:
            parsed = json.loads(candidate_json)
            normalized = {str(k).strip().upper(): str(v).strip().upper() for k, v in parsed.items()}
            return normalized, True, "Parsed from fallback JSON object"
        except:
            pass

    # try 4: extract pair lines like:
    # M1: W10
    # - M2: W8
    # M3-W9
    pair_matches = re.findall(r"-?\s*(M\d+)\s*[:\-]\s*(W\d+)", raw_output, re.IGNORECASE)

    if pair_matches:
        parsed = {}
        for m, w in pair_matches:
            parsed[m.strip().upper()] = w.strip().upper()

        if len(parsed) == 10:
            ordered = {f"M{i}": parsed[f"M{i}"] for i in range(1, 11) if f"M{i}" in parsed}
            if len(ordered) == 10:
                return ordered, True, "Parsed from pair lines"

        return parsed, True, "Parsed partial matching from pair lines"

    # try 5: extract lines like "M1 with W10"
    with_matches = re.findall(r"(M\d+)\s+with\s+(W\d+)", raw_output, re.IGNORECASE)

    if with_matches:
        parsed = {}
        for m, w in with_matches:
            parsed[m.strip().upper()] = w.strip().upper()

        if len(parsed) == 10:
            ordered = {f"M{i}": parsed[f"M{i}"] for i in range(1, 11) if f"M{i}" in parsed}
            if len(ordered) == 10:
                return ordered, True, "Parsed from 'M with W' lines"

        return parsed, True, "Parsed partial matching from 'M with W' lines"

    return None, False, "No JSON or matching pairs found"

In [ ]:
def process_model_response(raw_output, men_prefs, women_prefs, ground_truth_string, expected_size=10):
    parsed_matching, parsed_ok, parse_msg = extract_matching_from_response(raw_output)

    if not parsed_ok:
        result = {
            "parsed_ok": False,
            "parse_msg": parse_msg,
            "parsed_matching": None,
            "evaluation": None
        }

        print("Parsing failed:", parse_msg)
        return result

    evaluation = evaluate_matching(
        parsed_matching,
        men_prefs,
        women_prefs,
        ground_truth_string,
        expected_size=expected_size
    )

    print_evaluation_summary(evaluation)

    return {
        "parsed_ok": True,
        "parse_msg": parse_msg,
        "parsed_matching": parsed_matching,
        "evaluation": evaluation
    }

In [ ]:
# =========================================
# phase 3 model 1 basic llm
# =========================================


# =========================================
# step 1 initialize basic model
# =========================================
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",  # fast + free
    temperature=0
)

response = llm.invoke(prompt)

print(response.content)

<answer>
{
  "M1": "W10",
  "M2": "W8",
  "M3": "W4",
  "M4": "W2",
  "M5": "W3",
  "M6": "W7",
  "M7": "W6",
  "M8": "W5",
  "M9": "W9",
  "M10": "W1"
}
</answer>


In [ ]:

# =========================================
# step 2 evaluate basic model output
# =========================================

basic_model_result = process_model_response(
    response.content,
    men_prefs,
    women_prefs,
    ground_truth,
    expected_size=10
)

print(basic_model_result)

parsed: True
valid: True
stable: False
exact match: False
validity message: valid one-to-one matching
blocking pairs count: 3
blocking pairs: [('M9', 'W4'), ('M9', 'W1'), ('M10', 'W2')]
{'parsed_ok': True, 'parse_msg': 'Parsed from <answer> tags', 'parsed_matching': {'M1': 'W10', 'M2': 'W8', 'M3': 'W4', 'M4': 'W2', 'M5': 'W3', 'M6': 'W7', 'M7': 'W6', 'M8': 'W5', 'M9': 'W9', 'M10': 'W1'}, 'evaluation': {'parsed_successfully': True, 'is_valid': True, 'is_stable': False, 'blocking_pairs': [('M9', 'W4'), ('M9', 'W1'), ('M10', 'W2')], 'exact_match': False, 'ground_truth_dict': {'M1': 'W10', 'M2': 'W8', 'M3': 'W9', 'M4': 'W6', 'M5': 'W3', 'M6': 'W7', 'M7': 'W1', 'M8': 'W5', 'M9': 'W4', 'M10': 'W2'}, 'validity_message': 'valid one-to-one matching'}}


In [ ]:
# =========================================
# phase 4 model 2 reasoning llm
# =========================================


# =========================================
# step 1 initialize reasoning  model
# =========================================
from langchain_groq import ChatGroq

llm_reasoning = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

response_reasoning = llm_reasoning.invoke(prompt)

print(response_reasoning.content)

<answer>
{
  "M1": "W10",
  "M2": "W8",
  "M3": "W9",
  "M4": "W2",
  "M5": "W5",
  "M6": "W7",
  "M7": "W6",
  "M8": "W3",
  "M9": "W1",
  "M10": "W4"
}
</answer>


In [ ]:
# =========================================
# step 2 evaluate reasoning model output
# =========================================

reasoning_model_result = process_model_response(
    response_reasoning.content,
    men_prefs,
    women_prefs,
    ground_truth,
    expected_size=10
)

print(reasoning_model_result)

parsed: True
valid: True
stable: False
exact match: False
validity message: valid one-to-one matching
blocking pairs count: 8
blocking pairs: [('M3', 'W4'), ('M5', 'W3'), ('M8', 'W5'), ('M9', 'W4'), ('M10', 'W2'), ('M10', 'W3'), ('M10', 'W7'), ('M10', 'W10')]
{'parsed_ok': True, 'parse_msg': 'Parsed from <answer> tags', 'parsed_matching': {'M1': 'W10', 'M2': 'W8', 'M3': 'W9', 'M4': 'W2', 'M5': 'W5', 'M6': 'W7', 'M7': 'W6', 'M8': 'W3', 'M9': 'W1', 'M10': 'W4'}, 'evaluation': {'parsed_successfully': True, 'is_valid': True, 'is_stable': False, 'blocking_pairs': [('M3', 'W4'), ('M5', 'W3'), ('M8', 'W5'), ('M9', 'W4'), ('M10', 'W2'), ('M10', 'W3'), ('M10', 'W7'), ('M10', 'W10')], 'exact_match': False, 'ground_truth_dict': {'M1': 'W10', 'M2': 'W8', 'M3': 'W9', 'M4': 'W6', 'M5': 'W3', 'M6': 'W7', 'M7': 'W1', 'M8': 'W5', 'M9': 'W4', 'M10': 'W2'}, 'validity_message': 'valid one-to-one matching'}}


In [ ]:
# =========================================
# phase 5 model 3 advanced llm
# =========================================

!pip install -q together

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.7/344.7 kB 17.8 MB/s eta 0:00:00


In [ ]:
import os
from getpass import getpass
from together import Together

if "TOGETHER_API_KEY" not in os.environ:
    os.environ["TOGETHER_API_KEY"] = getpass("Enter Together API key: ")

together_client = Together(api_key=os.environ["TOGETHER_API_KEY"])

Enter Together API key: ··········


In [ ]:
response = together_client.chat.completions.create(
    model="deepseek-ai/DeepSeek-R1",
    messages=[
        {"role": "user", "content": "Say hello in one sentence."}
    ],
    temperature=0,
    max_tokens=200
)

print(response)
print("-----")
print(response.choices[0])
print("-----")
print(response.choices[0].message)

ChatCompletion(id='of9rVQN-2dTqGa-9ec5c08cee9e823b', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChoiceMessage(content='Hello! How can I assist you today? 😊', role='assistant', function_call=None, reasoning='Okay, the user just asked me to "say hello in one sentence." That\'s a straightforward request. \n\nHmm, they didn\'t specify any language or context, so I\'ll keep it simple and friendly. A classic "Hello!" feels too abrupt though... maybe add a tiny bit of warmth? \n\n*Pauses* Wait, they said "one sentence" - better not overcomplicate it. Just a cheerful greeting with a subtle offer of help in case they need more. \n\n*Types* There we go - short, clear, and leaves the door open if they actually want something else. Hope they\'re having a good day wherever they are.\n', tool_calls=[]), seed=None, text=None, top_logprobs=None)], created=1776201603, model='deepseek-ai/DeepSeek-R1', object='chat.completion', prompt=[], usage=ChatCompletionUsage(completion_to

In [ ]:
prompt_advanced = f"""
You are solving a stable matching problem.

There are 10 men and 10 women.

Men preferences:
{men_text}

Women preferences:
{women_text}

Find the proposer-optimal stable matching.

You may reason step by step, but at the VERY END you must output exactly one final line in this format:

FINAL_MATCHING: M1-W?, M2-W?, M3-W?, M4-W?, M5-W?, M6-W?, M7-W?, M8-W?, M9-W?, M10-W?

Do not use JSON.
Do not use <answer> tags.
Make sure the FINAL_MATCHING line appears exactly once at the end.
"""

In [ ]:
def get_together_advanced_response(prompt, model_name="deepseek-ai/DeepSeek-R1"):
    response = together_client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0,
        max_tokens=4000
    )

    message = response.choices[0].message

    content = getattr(message, "content", "") or ""
    reasoning = getattr(message, "reasoning", "") or ""

    full_text = ""
    if reasoning.strip():
        full_text += reasoning.strip()
    if content.strip():
        if full_text:
            full_text += "\n\n"
        full_text += content.strip()

    if isinstance(full_text, list):
        parts = []
        for item in full_text:
            if isinstance(item, dict) and "text" in item:
                parts.append(item["text"])
            else:
                parts.append(str(item))
        full_text = "\n".join(parts)

    return str(full_text).strip()

In [ ]:
advanced_raw_output = get_together_advanced_response(prompt_advanced)
print(advanced_raw_output[-1500:])

3.

M7 is proposing, M7 is rank 10? Last.

Rank 10 is worse than rank 3, so W4 prefers M9 over M7, so she rejects M7.

M7 rejected again.

Now M7 needs to propose to next woman. Next after W4 is W1.

So M7 proposes to W1.

W1 is currently free? Not engaged.

Engagements: W10-M1, W8-M2, W4-M9, W3-M5, W7-M6, W6-M4, W2-M10, W5-M8, W9-M3

W1 not engaged, so W1 accepts M7 tentatively.

Engagement: M7 - W1

Now all men are engaged? Let's see:

M1-W10, M2-W8, M3-W9, M4-W6, M5-W3, M6-W7, M7-W1, M8-W5, M9-W4, M10-W2

All women: W1-M7, W2-M10, W3-M5, W4-M9, W5-M8, W6-M4, W7-M6, W8-M2, W9-M3, W10-M1

All seem engaged. But is this stable? We need to check if it's stable, but in Gale-Shapley, it should be stable, but proposer-optimal.

But let me verify if any man can do better or if there are blocking pairs.

Since we're doing men-proposing, this should be men-optimal.

But let me confirm the matches.

M1 with W10

M2 with W8

M3 with W9

M4 with W6

M5 with W3

M6 with W7

M7 with W1

M8 with W5


In [ ]:
advanced_model_result = process_model_response(
    advanced_raw_output,
    men_prefs,
    women_prefs,
    ground_truth,
    expected_size=10
)

print(advanced_model_result)

parsed: True
valid: True
stable: True
exact match: True
validity message: valid one-to-one matching
blocking pairs count: 0
{'parsed_ok': True, 'parse_msg': 'Parsed from pair lines', 'parsed_matching': {'M1': 'W10', 'M2': 'W8', 'M3': 'W9', 'M4': 'W6', 'M5': 'W3', 'M6': 'W7', 'M7': 'W1', 'M8': 'W5', 'M9': 'W4', 'M10': 'W2'}, 'evaluation': {'parsed_successfully': True, 'is_valid': True, 'is_stable': True, 'blocking_pairs': [], 'exact_match': True, 'ground_truth_dict': {'M1': 'W10', 'M2': 'W8', 'M3': 'W9', 'M4': 'W6', 'M5': 'W3', 'M6': 'W7', 'M7': 'W1', 'M8': 'W5', 'M9': 'W4', 'M10': 'W2'}, 'validity_message': 'valid one-to-one matching'}}


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = ["LLaMA 3.1\n8B (Basic)", "LLaMA 3.3\n70B (Reasoning)", "DeepSeek-R1\n(Advanced)"]

metrics = {
    "Valid":       [1, 1, 1],
    "Stable":      [0, 0, 1],
    "Exact Match": [0, 0, 1],
}

x = np.arange(len(models))
width = 0.25
colors = ["#4C72B0", "#55A868", "#C44E52"]

fig, ax = plt.subplots(figsize=(10, 5))
for i, (metric, values) in enumerate(metrics.items()):
    ax.bar(x + i * width, values, width, label=metric, color=colors[i])

ax.set_xticks(x + width)
ax.set_xticklabels(models, fontsize=11)
ax.set_yticks([0, 1])
ax.set_yticklabels(["No", "Yes"], fontsize=11)
ax.set_title("Model Comparison: Validity, Stability, Exact Match", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.set_ylim(0, 1.4)
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("comparison_bar.png", dpi=150)
plt.show()

In [ ]:
models = ["LLaMA 3.1 8B\n(Basic)", "LLaMA 3.3 70B\n(Reasoning)", "DeepSeek-R1\n(Advanced)"]
blocking_counts = [3, 8, 0]
colors = ["#e07b54", "#e07b54", "#55A868"]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(models, blocking_counts, color=colors, edgecolor="black", width=0.5)

for bar, val in zip(bars, blocking_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(val), ha="center", va="bottom", fontsize=12, fontweight="bold")

ax.set_title("Number of Blocking Pairs per Model\n(Lower is Better)", fontsize=13, fontweight="bold")
ax.set_ylabel("Blocking Pairs", fontsize=11)
ax.set_ylim(0, 11)
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("blocking_pairs.png", dpi=150)
plt.show()

In [ ]:
basic_matching      = {'M1':'W10','M2':'W8','M3':'W4','M4':'W2','M5':'W3','M6':'W7','M7':'W6','M8':'W5','M9':'W9','M10':'W1'}
reasoning_matching  = {'M1':'W10','M2':'W8','M3':'W9','M4':'W2','M5':'W5','M6':'W7','M7':'W6','M8':'W3','M9':'W1','M10':'W4'}
advanced_matching   = {'M1':'W10','M2':'W8','M3':'W9','M4':'W6','M5':'W3','M6':'W7','M7':'W1','M8':'W5','M9':'W4','M10':'W2'}
ground_truth_dict   = {'M1':'W10','M2':'W8','M3':'W9','M4':'W6','M5':'W3','M6':'W7','M7':'W1','M8':'W5','M9':'W4','M10':'W2'}

men   = [f"M{i}" for i in range(1, 11)]
women = [f"W{i}" for i in range(1, 11)]

all_matchings = {
    "Basic\n(LLaMA 8B)":     basic_matching,
    "Reasoning\n(LLaMA 70B)": reasoning_matching,
    "Advanced\n(DeepSeek-R1)": advanced_matching,
    "Ground\nTruth":          ground_truth_dict,
}

fig, axes = plt.subplots(1, 4, figsize=(16, 6))

for ax, (title, matching) in zip(axes, all_matchings.items()):
    grid = np.zeros((10, 10))
    for m, w in matching.items():
        mi = int(m[1:]) - 1
        wi = int(w[1:]) - 1
        grid[mi][wi] = 1

    ax.imshow(grid, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(10))
    ax.set_xticklabels(women, fontsize=7, rotation=45)
    ax.set_yticks(range(10))
    ax.set_yticklabels(men, fontsize=7)
    ax.set_title(title, fontsize=10, fontweight="bold")

plt.suptitle("Matching Heatmap: Which Man Was Matched to Which Woman", fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("matching_heatmap.png", dpi=150)
plt.show()

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# list of saved images
images = ["comparison_bar.png", "blocking_pairs.png", "matching_heatmap.png"]
titles = ["Visual 1: Model Comparison", "Visual 2: Blocking Pairs", "Visual 3: Matching Heatmap"]

with PdfPages("model_visuals.pdf") as pdf:
    for img_path, title in zip(images, titles):
        fig, ax = plt.subplots(figsize=(11, 8))
        img = mpimg.imread(img_path)
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(title, fontsize=14, fontweight="bold", pad=10)
        pdf.savefig(fig, bbox_inches="tight")
        plt.close()

print("PDF saved as model_visuals.pdf")